<a href="https://colab.research.google.com/github/priyaaaww/ITA06_LAB_EXP/blob/main/EXP_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import math
from collections import Counter

df = pd.read_csv("/content/play_tennis.csv")
df.columns = df.columns.str.strip().str.lower()

# Remove day/index columns
df = df.drop(columns=["day"], errors="ignore")

target = "play"
features = [c for c in df.columns if c != target]

def entropy(y):
    counts = Counter(y)
    total = len(y)
    return -sum(
        (n / total) * math.log2(n / total)
        for n in counts.values()
    )

def information_gain(df, feature, target):
    total = entropy(df[target])
    weighted = 0

    for value in df[feature].unique():
        subset = df[df[feature] == value]
        weighted += len(subset) / len(df) * entropy(subset[target])

    return total - weighted

def id3(df, features, target):
    if df[target].nunique() == 1:
        return df[target].iloc[0]

    if not features:
        return df[target].mode()[0]

    best = max(
        features,
        key=lambda f: information_gain(df, f, target)
    )

    tree = {best: {}}

    for value in df[best].unique():
        subset = df[df[best] == value]
        remaining = [f for f in features if f != best]

        tree[best][value] = id3(subset, remaining, target)

    return tree

def predict(tree, sample):
    if not isinstance(tree, dict):
        return tree

    feature = next(iter(tree))
    value = sample[feature]

    if value not in tree[feature]:
        return "Unknown"

    return predict(tree[feature][value], sample)

tree = id3(df, features, target)

print("Decision Tree:")
print(tree)

sample = {
    "outlook": "sunny",
    "temp": "cool",
    "humidity": "high",
    "windy": True
}

print("Prediction:", predict(tree, sample))

Decision Tree:
{'outlook': {'Sunny': {'humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'wind': {'Weak': 'Yes', 'Strong': 'No'}}}}
Prediction: Unknown
